# 04 · Validate — both states from one sequence + energy gap + the benchmark

**Standard slot:** *validate (in silico).* **For Project 22 this is the core science:** confirm one
sequence can fold to **both** states (and worry that AF2 may only show one), characterize the
**energy-gap distribution**, and run the **single- vs multi-state benchmark** — the contrast that
turns this from a demo into a study (D3 part 2).

Needs `results/multistate_designs.csv` (from notebook 02). Mock backend runs anywhere.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · AF2 predicts BOTH states from one sequence

For trusted picks, predict the shared sequence toward **each** state and compare per-state scRMSD.
**The central caveat:** AF2 returns a single dominant state and may not capture both — so where
possible **bias prediction toward each state** (templates / initial guess) and report the unbiased
*and* state-biased results. A "switch" claimed from one unbiased model is not a switch.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import multistate_tools as ms

df = pd.read_csv("results/multistate_designs.csv")

# per-state foldability (uses the project's bars)
df["pass_a"] = (df["scrmsd_a"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_a"] >= ms.SWITCH_PLDDT)
df["pass_b"] = (df["scrmsd_b"] < ms.SELF_CONSISTENT_SCRMSD) & (df["plddt_b"] >= ms.SWITCH_PLDDT)
df["pass_both"] = df["pass_a"] & df["pass_b"]

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(df["scrmsd_a"], df["scrmsd_b"], c=df["pass_both"].map({True: "tab:green", False: "tab:gray"}))
ax.axvline(ms.SELF_CONSISTENT_SCRMSD, ls="--", c="k", lw=0.8)
ax.axhline(ms.SELF_CONSISTENT_SCRMSD, ls="--", c="k", lw=0.8)
ax.set_xlabel("state A scRMSD (A)"); ax.set_ylabel("state B scRMSD (A)")
ax.set_title("Per-state self-consistency (green = passes BOTH)")
plt.tight_layout(); plt.savefig("results/per_state_scrmsd.png", dpi=150); plt.show()
print("green points pass BOTH states; lower-left quadrant = the switch candidates.")
print("[SYNTHETIC EXAMPLE_DATA — AF2 may not actually produce both states; see the caveat]")

## 2 · The state energy-gap distribution

A switch needs the gap **inside the band**: close enough to interconvert, distinct enough for a
defined OFF/ON. Plot the distribution and shade the switchable band. **The gap is a teaching-grade
proxy in relative units — NOT a ΔΔG**; a "switchable" flag is a hypothesis, not a measurement.

In [ ]:
gaps = df["energy_gap"].dropna().values
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(gaps, bins=15, color="tab:blue", alpha=0.8)
ax.axvspan(ms.SWITCH_GAP_MIN, ms.SWITCH_GAP_MAX, color="tab:green", alpha=0.2,
           label=f"switchable band [{ms.SWITCH_GAP_MIN}, {ms.SWITCH_GAP_MAX}]")
ax.set_xlabel("state energy gap (relative units, NOT kcal/mol)")
ax.set_ylabel("designs"); ax.set_title("State energy-gap distribution")
ax.legend(); plt.tight_layout(); plt.savefig("results/energy_gap_dist.png", dpi=150); plt.show()
n_switch = int(df["switchable"].fillna(False).sum())
print(f"{n_switch} / {len(df)} designs fall inside the switchable band (mock).")
print("CAVEAT: relative-units proxy from prediction fit, not a free energy. Confirm with physics/MD.")

## 3 · Benchmark: single-state vs multi-state design (the result)

Design each backbone **alone** with normal (single-state) MPNN and show those single-state sequences
**fail the other state**. The contrast — multi-state sequences fit *both*, single-state ones fit only
their own — is the headline finding. (Mock: we approximate single-state design by scoring each
sequence against only its designed-for state.)

In [ ]:
# Multi-state: how often does ONE sequence pass BOTH states?
multi_both = float(df["pass_both"].mean())

# Single-state baseline (approximation on mock): design FOR state A only, then test state B.
# A single-state-A sequence is, by construction, optimized for A; we ask how often it ALSO passes B.
# (On real data: run normal MPNN on backbone A alone, predict that sequence toward B, measure pass_b.)
single_a_passes_b = float(df.loc[df["pass_a"], "pass_b"].mean()) if df["pass_a"].any() else float("nan")

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.bar(["multi-state\n(pass BOTH)", "single-state A\n(also passes B)"],
       [multi_both, single_a_passes_b], color=["tab:green", "tab:gray"])
ax.set_ylabel("fraction"); ax.set_title("Multi-state vs single-state: who satisfies both states?")
for i, v in enumerate([multi_both, single_a_passes_b]):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout(); plt.savefig("results/single_vs_multi.png", dpi=150); plt.show()
print(f"multi-state pass-both rate     : {multi_both:.2f}")
print(f"single-state-A also-passes-B   : {single_a_passes_b:.2f}")
print("Expected (and the point): single-state sequences usually FAIL the other state.")
print("[SYNTHETIC EXAMPLE_DATA — on real data, run normal MPNN per backbone for the true baseline]")

## 4 · Honest hit-rate ladder + the energy-gap caveat

Report the ladder explicitly and sanity-check the switchable picks: is the gap real, or an artifact
of AF2 only seeing one state? A switchable flag survives only if **both** states reproduce when you
bias prediction toward each.

In [ ]:
ladder = dict(
    N_total=len(df),
    N_pass_A=int(df["pass_a"].sum()),
    N_pass_B=int(df["pass_b"].sum()),
    N_pass_BOTH=int(df["pass_both"].sum()),
    N_switchable=int(df["switchable"].fillna(False).sum()),
    N_both_and_switchable=int((df["pass_both"] & df["switchable"].fillna(False)).sum()),
)
for k, v in ladder.items():
    print(f"  {k:24s}: {v}")
print("\nENERGY-GAP CAVEAT (state it in the report):")
print(" - the gap is a relative-units proxy from prediction fit, NOT a kcal/mol free energy;")
print(" - AF2 may only return one state, so a 'switchable' flag is a HYPOTHESIS to test in the wet lab;")
print(" - validate switchable picks by biasing prediction toward EACH state (templates/initial guess).")
print("\n[mock numbers are SYNTHETIC EXAMPLE_DATA]")

## 5 · (Extension) transition-plausibility MD `[extension]`
On a top pick, run a short OpenMM MD (10–50 ns) on each predicted state to check it stays put, and
(harder) probe whether A↔B is plausible. **Not** a free-energy calculation — a sanity probe.

In [ ]:
# Scaffold (fill in with the real OpenMM backend on a GPU runtime):
# 1) prep each predicted state PDB (pdbfixer: add H, solvate, neutralize) at the trigger condition;
# 2) minimize + equilibrate; run 10-50 ns; measure backbone RMSD drift vs the predicted state;
# 3) (stretch) attempt a biased path A->B and report whether the transition looks plausible.
print("Transition-MD scaffold — implement with OpenMM on a GPU runtime (see MANUAL.md §2). "
      "Keep runs short; this probes plausibility, not free energy.")

## D3 (part 2) checklist
- [ ] AF2 predicts **both** states from one sequence; per-state scRMSD plotted; state-biased predictions discussed.
- [ ] Energy-gap distribution with the switchable band; the **gap caveat** stated (not a ΔΔG; AF2 may miss a state).
- [ ] **Single- vs multi-state benchmark**: single-state sequences shown to fail the other state.
- [ ] Honest hit-rate ladder: N(A) / N(B) / N(BOTH) / N(switchable) / N(both & switchable).
- [ ] (ext) transition MD on a top pick.

**Next:** `05_validation_plan.ipynb` — the state-change read-out plan with controls.